# OpenAI · OpenRouter API 저비용 테스트

같은 질문을 두 API에 보내며 **요청 → 응답 → 토큰 사용량** 흐름을 확인합니다.

> 안전장치: 아래의 `RUN_OPENAI`, `RUN_OPENROUTER`는 기본값이 `False`입니다. API 호출 셀은 스위치를 직접 켜기 전까지 실행되지 않습니다.

## 1. 준비

저장소 루트의 `.env`에서 `OPENAI_API_KEY`와 `OPENROUTER_API_KEY`를 자동으로 읽습니다. `.env`가 없으면 노트북을 시작한 PowerShell의 환경변수를 사용합니다. 키를 이 노트북에 직접 입력하지 않습니다.

두 위치 어디에도 키가 없으면 아래 준비 셀에서 해당 키가 준비되지 않았다고 표시하며, 실제 호출 셀은 요청을 보내지 않습니다.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

search_start = Path.cwd().resolve()
env_path = next(
    (folder / ".env" for folder in (search_start, *search_start.parents) if (folder / ".env").is_file()),
    None,
)
if env_path is not None:
    load_dotenv(env_path, override=False)
has_openai_key = bool(os.getenv("OPENAI_API_KEY"))
has_openrouter_key = bool(os.getenv("OPENROUTER_API_KEY"))

print("설정 파일:", env_path if env_path else ".env 없음 — PowerShell 환경변수 확인")
print("OpenAI 키 준비:", has_openai_key)
print("OpenRouter 키 준비:", has_openrouter_key)
# 키의 실제 값은 출력하지 않습니다.

## 2. 공통 질문과 비용 안전장치

답변 길이를 짧게 제한해 비용을 낮춥니다. 처음에는 한 서비스씩만 켜서 실행하는 것을 권장합니다.

In [ ]:
PROMPT = "API가 무엇인지 비개발자에게 한국어 3문장으로 설명해줘."
MAX_OUTPUT_TOKENS = 150

OPENAI_MODEL = "gpt-5.6-luna"
OPENROUTER_MODEL = "openrouter/free"

RUN_OPENAI = False
RUN_OPENROUTER = False

print({
    "prompt": PROMPT,
    "openai_model": OPENAI_MODEL,
    "openrouter_model": OPENROUTER_MODEL,
    "max_output_tokens": MAX_OUTPUT_TOKENS,
})

## 3. OpenAI Responses API 호출

OpenAI의 Responses API로 텍스트를 생성합니다. `RUN_OPENAI = True`로 바꾼 경우에만 호출됩니다.

In [ ]:
RUN_OPENAI = True

In [ ]:
openai_result = None

if not RUN_OPENAI:
    print("건너뜀: RUN_OPENAI를 True로 바꾸면 호출합니다.")
elif not has_openai_key:
    print("건너뜀: OPENAI_API_KEY 환경변수가 없습니다.")
else:
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=PROMPT,
        max_output_tokens=MAX_OUTPUT_TOKENS,
    )
    usage = response.usage
    openai_result = {
        "provider": "OpenAI",
        "model": OPENAI_MODEL,
        "text": response.output_text,
        "input_tokens": getattr(usage, "input_tokens", None),
        "output_tokens": getattr(usage, "output_tokens", None),
    }
    print(response.output_text)
    print("토큰 사용량:", usage)

## 4. OpenRouter 무료 라우터 호출

OpenAI Python SDK의 접속 주소만 OpenRouter로 바꾸어 호출합니다. 무료 라우터는 요청마다 실제 선택 모델이 달라질 수 있습니다.

In [ ]:
RUN_OPENROUTER = True

In [ ]:
openrouter_result = None

if not RUN_OPENROUTER:
    print("건너뜀: RUN_OPENROUTER를 True로 바꾸면 호출합니다.")
elif not has_openrouter_key:
    print("건너뜀: OPENROUTER_API_KEY 환경변수가 없습니다.")
else:
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
    )
    response = client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[{"role": "user", "content": PROMPT}],
        max_tokens=MAX_OUTPUT_TOKENS,
    )
    usage = response.usage
    openrouter_result = {
        "provider": "OpenRouter",
        "model": response.model,
        "text": response.choices[0].message.content,
        "input_tokens": getattr(usage, "prompt_tokens", None),
        "output_tokens": getattr(usage, "completion_tokens", None),
    }
    print(response.choices[0].message.content)
    print("실제 선택 모델:", response.model)
    print("토큰 사용량:", usage)

## 5. 결과 비교

실행한 서비스만 표 형태로 요약합니다. 품질뿐 아니라 실제 선택 모델과 토큰 수도 함께 비교하세요.

In [ ]:
results = [result for result in (openai_result, openrouter_result) if result]

if not results:
    print("아직 실행된 API 호출이 없습니다. 위의 RUN 스위치를 하나씩 켜보세요.")
else:
    for result in results:
        print("-" * 70)
        print(f"{result['provider']} | {result['model']}")
        print(f"입력 {result['input_tokens']} 토큰 / 출력 {result['output_tokens']} 토큰")
        print(result["text"])

## 6. 다음 실험 아이디어

아래 프롬프트로 바꿔보면 API가 단순 채팅을 넘어 프로그램용 데이터 생성에 쓰이는 방식을 확인할 수 있습니다.

```text
다음 고객 문의를 JSON으로 정리해줘:
주문번호 1234인데 상품이 아직 도착하지 않았어요.
필드: category, order_id, sentiment, summary
```

주의: 무료 모델은 가용성과 출력 형식이 달라질 수 있습니다. 실제 업무 데이터나 개인정보는 입문 테스트에 넣지 마세요.